In [1]:
import hashlib
import json
import sys
from collections import Counter
from datetime import datetime, timezone
from itertools import combinations
from pathlib import Path

REPO_ROOT = Path.cwd().parent
DB_PATH = REPO_ROOT / "database" / "CSCAF_public.db"
OUTPUT_FOLDER = REPO_ROOT / "methodology" / "output"

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

assert DB_PATH.exists(), "run data_load/01_load_ecsf_to_database.ipynb first"

from cscaf.storage import Storage
from cscaf.study import Study

study = Study.from_config()
storage = Storage.of(study)
print("database:", DB_PATH.relative_to(REPO_ROOT))

database: database\CSCAF_public.db


In [2]:
framework = study.section("reference_framework")
framework_file = REPO_ROOT / framework["file"]
checksum = hashlib.sha256(framework_file.read_bytes()).hexdigest()
print("file    :", framework_file.relative_to(REPO_ROOT))
print("sha256  :", checksum)
print("expected:", framework["sha256"])
assert checksum == framework["sha256"]

file    : raw_data\referent_framework\ECSF_v1.json
sha256  : 7729813fb61957349913dfdabd7b1db71bec2d3862aa79c5095315e366e58502
expected: 7729813fb61957349913dfdabd7b1db71bec2d3862aa79c5095315e366e58502


In [3]:
rows = storage.framework.role_statements()
full_names = {short: full for short, full, _, _, _ in rows}
texts = {sid: text for _, _, _, sid, text in rows}
roles = {}
kind_rows = Counter()
kind_statements = {}
for role, _, kind, sid, _ in rows:
    roles.setdefault(role, set()).add(sid)
    kind_rows[kind] += 1
    kind_statements.setdefault(kind, set()).add(sid)

statements = set().union(*roles.values())
assert not (kind_statements["skills"] & kind_statements["knowledge"])
units = {
    "roles": len(roles),
    "role_statement_pairs": len(rows),
    "skill_pairs": kind_rows["skills"],
    "knowledge_pairs": kind_rows["knowledge"],
    "statements": len(statements),
    "skill_statements": len(kind_statements["skills"]),
    "knowledge_statements": len(kind_statements["knowledge"]),
}
assert units["role_statement_pairs"] == framework["expected_rows"]
assert units["statements"] == framework["expected_statements"]
units

{'roles': 12,
 'role_statement_pairs': 227,
 'skill_pairs': 103,
 'knowledge_pairs': 124,
 'statements': 150,
 'skill_statements': 83,
 'knowledge_statements': 67}

In [4]:
profile_size = {role: len(ids) for role, ids in roles.items()}
roles_per_statement = Counter(sid for ids in roles.values() for sid in ids)
most_shared = max(roles_per_statement.values())
profiles = {
    "smallest_profile": min(profile_size.values()),
    "largest_profile": max(profile_size.values()),
    "single_role_statements": sum(1 for n in roles_per_statement.values() if n == 1),
    "shared_statements": sum(1 for n in roles_per_statement.values() if n > 1),
    "most_shared_statements": sorted(texts[sid] for sid, n in roles_per_statement.items() if n == most_shared),
    "most_shared_roles": most_shared,
}
for role in sorted(profile_size, key=profile_size.get):
    print(f"{full_names[role]:45s} {profile_size[role]:3d}")
profiles

Cybersecurity Researcher                       12
Cyber Legal, Policy & Compliance Officer       13
Cybersecurity Auditor                          16
Cybersecurity Risk Manager                     16
Cybersecurity Educator                         17
Cybersecurity Implementer                      18
Digital Forensics Investigator                 18
Cyber Incident Responder                       19
Cyber Threat Intelligence Specialist           23
Penetration Tester                             23
Cybersecurity Architect                        25
Chief Information Security Officer (CISO)      27


{'smallest_profile': 12,
 'largest_profile': 27,
 'single_role_statements': 121,
 'shared_statements': 29,
 'most_shared_statements': ['Cybersecurity-related certifications'],
 'most_shared_roles': 9}

In [5]:
shares = {}
for role, ids in sorted(roles.items()):
    others = set().union(*(other for name, other in roles.items() if name != role))
    shared = len(ids & others)
    shares[role] = {"statements": len(ids), "shared": shared, "share": shared / len(ids)}
    print(f"{full_names[role]:45s} {shared:3d} of {len(ids):3d}  {shared / len(ids):.3f}")

mean_share = sum(v["share"] for v in shares.values()) / len(shares)
lowest = min(shares, key=lambda r: shares[r]["share"])
highest = max(shares, key=lambda r: shares[r]["share"])
pairs = {f"{a}|{b}": len(roles[a] & roles[b]) for a, b in combinations(sorted(roles), 2)}
overlap = {
    "mean_share": mean_share,
    "lowest_role": lowest,
    "lowest_share": shares[lowest]["share"],
    "highest_role": highest,
    "highest_share": shares[highest]["share"],
    "architect_statements": shares["Architect"]["statements"],
    "architect_shared": shares["Architect"]["shared"],
    "architect_with_risk_manager": len(roles["Architect"] & roles["RiskManager"]),
    "architect_with_ciso": len(roles["Architect"] & roles["CISO"]),
}
print()
for pair, n in sorted(pairs.items(), key=lambda item: -item[1])[:8]:
    print(f"{pair:28s} {n}")
print()
print("Architect and CISO share:")
for sid in sorted(roles["Architect"] & roles["CISO"]):
    print("  ", texts[sid])
overlap

Cybersecurity Architect                        11 of  25  0.440
Cybersecurity Auditor                           6 of  16  0.375
Chief Information Security Officer (CISO)       9 of  27  0.333
Cybersecurity Educator                          7 of  17  0.412
Digital Forensics Investigator                  8 of  18  0.444
Cybersecurity Implementer                      11 of  18  0.611
Cyber Incident Responder                        9 of  19  0.474
Cyber Legal, Policy & Compliance Officer        5 of  13  0.385
Penetration Tester                             11 of  23  0.478
Cybersecurity Researcher                        7 of  12  0.583
Cybersecurity Risk Manager                      9 of  16  0.562
Cyber Threat Intelligence Specialist           13 of  23  0.565

Forensics|IncidentResponder  7
IncidentResponder|ThreatIntel 7
PenTester|ThreatIntel        7
Architect|RiskManager        6
Implementer|PenTester        6
Implementer|ThreatIntel      6
IncidentResponder|PenTester  6
Architect|Edu

{'mean_share': 0.47193873200965336,
 'lowest_role': 'CISO',
 'lowest_share': 0.3333333333333333,
 'highest_role': 'Implementer',
 'highest_share': 0.6111111111111112,
 'architect_statements': 25,
 'architect_shared': 11,
 'architect_with_risk_manager': 6,
 'architect_with_ciso': 3}

In [6]:
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
output = OUTPUT_FOLDER / "01_reference_framework.json"
record = {
    "created_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "paper_section": "3.1",
    "framework_file": framework["file"],
    "framework_sha256": checksum,
    "units": units,
    "profiles": profiles,
    "overlap": overlap,
    "role_shares": shares,
    "role_pairs": pairs,
}
output.write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8")
print("written to", output.relative_to(REPO_ROOT))

written to methodology\output\01_reference_framework.json
